# Notebook 3: Dynamic Depot & Fleet Optimization Engine

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

orders = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Twiga Data Science/twiga_routing_model/orders.csv')
shops = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Twiga Data Science/twiga_routing_model/shops.csv')
depots = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Twiga Data Science/twiga_routing_model/depots-master.csv')
routes = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Twiga Data Science/twiga_routing_model/routes.csv')
vehicles = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Twiga Data Science/twiga_routing_model/trucks-vehicles.csv')


## Depot Demand

In [ ]:

orders_enriched = orders.merge(depots,on='depot_id',how='left')

depot_demand = (
    orders_enriched
    .groupby('depot_name')
    .agg(
        total_weight=('weight','sum'),
        orders=('order_id','count'),
        shops=('shop_id','nunique')
    )
    .reset_index()
)

depot_demand.sort_values('total_weight',ascending=False).head(20)


## Depot Pressure Score

In [ ]:

depot_demand['pressure_score'] = (
    depot_demand['total_weight']/depot_demand['total_weight'].max()*100
)

depot_demand.sort_values('pressure_score',ascending=False)


## Truck Recommendations

In [ ]:

TARGET_DROPS = 60

def recommend(weight, shops):
    trucks_45 = int(np.floor(weight/4500))
    remaining = max(weight - trucks_45*4500,0)
    trucks_25 = int(np.ceil(remaining/2500))

    total_trucks = max(trucks_45+trucks_25,1)
    drops_per_truck = shops/total_trucks

    return pd.Series([trucks_45,trucks_25,total_trucks,drops_per_truck])

depot_demand[['trucks_4_5T','trucks_2_5T','total_trucks','drops_per_truck']] = (
    depot_demand.apply(
        lambda x: recommend(x.total_weight,x.shops),
        axis=1
    )
)

depot_demand.sort_values('total_weight',ascending=False)


## Dynamic Depot Split Recommendations

In [ ]:

SPLIT_WEIGHT = 60000

depot_demand['recommended_operational_depots'] = np.where(
    depot_demand['total_weight'] > SPLIT_WEIGHT,
    np.ceil(depot_demand['total_weight']/SPLIT_WEIGHT),
    1
)

depot_demand[[
    'depot_name',
    'total_weight',
    'recommended_operational_depots'
]].sort_values('total_weight',ascending=False)


## Depot Demand Chart

In [ ]:

plot_df = depot_demand.sort_values('total_weight',ascending=False)

plt.figure(figsize=(12,6))
plt.bar(plot_df['depot_name'],plot_df['total_weight'])
plt.xticks(rotation=45)
plt.title('Depot Demand by Weight')
plt.tight_layout()
plt.show()


## Drop Density

In [ ]:

plt.figure(figsize=(10,5))
plt.hist(depot_demand['drops_per_truck'])
plt.title('Drops Per Truck Distribution')
plt.show()


## Final Recommendation Table

In [ ]:

depot_demand[[
    'depot_name',
    'total_weight',
    'shops',
    'trucks_4_5T',
    'trucks_2_5T',
    'total_trucks',
    'drops_per_truck',
    'recommended_operational_depots'
]].sort_values('total_weight',ascending=False)
